# PRF Restoration Reimplementation Guide

This branch intentionally excludes image restoration. This notebook documents the restoration work that was implemented later on `feature/prf-projection` so that a future team can reintroduce it cleanly if needed.

Primary restoration commits from the other branch:

- `32d7402` - Add PRF restoration imagery products
- `1295fa4` - Move PRF workflow to imagery actions
- `d354d2c` - Improve PRF restoration stability


## 1. Problem Framing

The PRF fitting work estimates a discrete pixel response function from detections. Restoration uses that estimated kernel to produce a second, non-destructive imagery product that is sharper than the distorted source.

Design decisions that worked well:

- Keep the original imagery untouched.
- Create a second imagery product named `<source> - PRF Restored`.
- Store restoration metadata in HDF5 alongside the restored imagery.
- Make restoration an explicit action from the Imagery tab instead of tying it to Map View.


## 2. Theory

Two restoration methods were implemented.

**Wiener/Tikhonov**

This is a regularized inverse filter in the Fourier domain:

$$\hat{X}(f)=\frac{H^*(f)}{|H(f)|^2 + \lambda}Y(f)$$

where `H` is the PRF optical transfer function and `lambda` is the regularization strength. Larger `lambda` is more conservative and suppresses ringing/noise amplification.

**Richardson-Lucy**

This is an iterative nonnegative deconvolution update:

$$x_{k+1} = x_k \cdot \left(\frac{y}{x_k * h} * h^{flip}\right)$$

It can recover point-source sharpness better, but it is more sensitive to noise, boundary artifacts, and nonnegative-input assumptions.


## 3. Files Added or Modified

The restoration implementation lived primarily in these files:

- `vista/algorithms/imagery/prf_restoration.py`
- `vista/widgets/core/imagery_viewer.py`
- `vista/widgets/core/data/imagery_panel.py`
- `vista/widgets/core/settings_dialog.py`
- `vista/imagery/imagery.py`
- `vista/widgets/core/data/data_loader.py`
- `PRF_PATCH.md`

The key architectural split was:

- `fit_prf_for_imagery(...)` for kernel estimation
- `create_prf_restored_imagery(...)` for derived imagery generation


In [ ]:
# Core restoration API shape from the implementation branch

RESTORATION_METHODS = ("Wiener/Tikhonov", "Richardson-Lucy")
RESTORATION_PRESETS = ("Conservative", "Balanced", "Aggressive", "Custom")

def restore_imagery_with_prf(images, prf_model, method, preset, custom_regularization, rl_iterations):
    """Return a restored image stack plus restoration metadata."""
    ...

def create_prf_restored_imagery(imagery, settings):
    """Create or reuse '<source> - PRF Restored' as a second imagery product."""
    ...


## 4. Stability Improvements That Mattered

The first restoration pass worked but produced visible darkening and edge ringing on the Golden Gate dataset. The later stabilization patch made three practical changes:

1. **Reflected padding before FFT deconvolution**
   This reduced wraparound edge artifacts.

2. **Background-aware restoration**
   A smooth background estimate was subtracted before restoration so that deconvolution focused on the residual source component rather than aggressively sharpening map texture.

3. **Residual mean preservation**
   The residual source mean was preserved before adding the background back so the restored image would not look darker overall.

The stabilized Wiener presets were:

- Conservative: `1.0`
- Balanced: `0.2`
- Aggressive: `0.05`


In [ ]:
# Key logic used in the stabilized Wiener/Tikhonov path

def restore_frame_wiener(image, kernel, regularization):
    background = smooth_background(image, kernel)
    source = image - background
    padded_source = reflect_pad(source)
    H = otf_from_kernel(kernel, padded_source.shape)
    restored_source = ifft2(fft2(padded_source) * conj(H) / (abs(H)**2 + regularization))
    restored_source = crop_padding(restored_source)
    restored_source += source.mean() - restored_source.mean()
    return restored_source + background

# Key logic used in the Richardson-Lucy path

def restore_frame_richardson_lucy(image, kernel, iterations):
    background_floor = percentile(image, 1.0)
    source = maximum(image - background_floor, 1e-6)
    estimate = reflect_pad(source)
    for _ in range(iterations):
        blurred = convolve(estimate, kernel)
        estimate *= convolve(source / maximum(blurred, 1e-6), flipped(kernel))
        estimate = maximum(estimate, 0.0)
    return crop_padding(estimate) + background_floor


## 5. UI and Data Flow

The final UI flow was intentionally explicit:

- `Settings -> Imagery` configured PRF and restoration parameters.
- `Data Manager -> Imagery -> Fit PRF` fit PRF metadata for the selected imagery.
- `Data Manager -> Imagery -> Create PRF Restored` created the derived imagery product.
- `File -> Save Imagery (HDF5)` exported either the original, the restored product, or both.

This was better than triggering restoration from Map View because restoration is expensive, scientifically meaningful, and should be under direct user control.


## 6. Verification Results and Lessons

Synthetic verification datasets were essential. The main datasets created were:

- NYC truth/distorted pair
- Golden Gate truth/distorted pair with real geographic texture
- Eastern US Airy Disk truth/distorted pair
- Eastern US Moffat truth/distorted pair

Important findings:

- PRF fitting behaved meaningfully on the controlled datasets and recovered the injected model family and orientation well enough to validate the fit path.
- Full-frame image restoration and point-source restoration do not optimize the same objective.
- Safer Wiener regularization improved full-frame stability on the Golden Gate dataset.
- Richardson-Lucy improved point-source chip sharpness more aggressively, but at the cost of greater numerical risk and worse whole-frame behavior.

Bottom line: PRF fitting reached a solid prototype state; restoration remained an active refinement area rather than a finished scientific product.


## 7. Recommended Reintroduction Plan

If a future team wants restoration back, the practical order is:

1. Re-add `vista/algorithms/imagery/prf_restoration.py`.
2. Re-add restoration metadata in HDF5 save/load.
3. Re-add `Fit PRF` / `Create PRF Restored` imagery actions.
4. Revalidate on the Golden Gate truth/distorted pair first.
5. Decide explicitly whether the optimization target is:
   whole-frame fidelity,
   point-source recovery,
   or downstream measurement quality.

That decision changes what “good restoration” means and should drive preset tuning and future algorithm work.
